## 6. Exception Handling

Programs <span style="color: rgba(255, 0, 0, 1);">**fail.**</span> A file is missing, a user types letters where a number was expected, a network goes down, a dictionary does not have the key we assumed... **Exceptions** are Python's mechanism to **signal** that something went wrong at one point of the program and to **decide**, possibly somewhere else, what to do about it.

This section goes from the mechanics to the design decisions:

* **1. Fundamentals**: what an exception is, how to read a traceback, and the full `try` / `except` / `else` / `finally` statement.
* **2. The exception hierarchy**: exceptions are **classes** organized by inheritance. That hierarchy determines what each `except` catches.
* **3. Raising and propagating**: `raise`, how exceptions travel up the call stack, where to catch them, re-raising and chaining.
* **4. Custom exceptions**: your own exception classes and hierarchies.
* **5. Style: EAFP vs LBYL**: "ask for forgiveness" vs "look before you leap".

### 6.1 Exception Fundamentals

#### **Errors and Exceptions**

In Python, there are (at least) two different kinds of errors: ***syntax errors*** and ***exceptions*.**

##### **Syntax Errors**

Syntax errors (aka parsing errors) occur when the *parser* detects an incorrect **statement,** **before** anything runs, so not a single line is executed. They are fixed by directly editing the code, they cannot be handled as exceptions.

The parser repeats the offending line and displays little arrows pointing at the place where the error was detected. For example:

In [ ]:
# Syntax error: the parser rejects the code before running anything.

while True print('Hello world')  # Missing ":"

##### **Exceptions**

Exceptions occur when the code is **valid,** but something goes wrong **while it runs**. They are raised at a specific line, so all the previous lines have already been executed. Even if a statement or expression is syntactically correct, it may cause an error when an attempt is made to **execute** it.

The last line of the error message indicates what happened. Python details what ***type*** of exception error it encountered. Exceptions come in different types, and the type is printed as part of the message, for example:

In [ ]:
# Exception: the code is valid, the error appears while running.

10 * (1/0)  # "ZeroDivisionError"

In [ ]:
a + 1  # "NameError"

In [ ]:
"1" + 1  # "TypeError"

The rest of the line provides detail based on the type of exception and what caused it. The preceding part of the error message shows the context where the exception occurred, in the form of a stack **traceback.**

Python comes with various **built-in** exceptions as well as the possibility to create user-defined exceptions.

##### **An Exception Is Raised**

When a line fails, Python:

1. Creates an **exception object**: an instance of a class (`ZeroDivisionError`, `KeyError`...) that carries information about the error (message, location).
2. **Interrupts** the normal flow at that point: the rest of the block does not run.
3. **Looks for a handler**: an **`except`** clause that matches the exception type. First in the current function, then in the function that called it, and so on upwards.
4. Then, one of two possibilities:
    * **Caught (handled) exception**: a matching `except` is found. Its block runs, and the program continues normally **after** that `try` statement.
    * **Uncaught exception**: no handler is found all the way up to the top level. Python prints the **traceback** and terminates the program. (In Jupyter, only the cell stops; the kernel and its variables survive.)

In [ ]:
def divide(a, b):
    return a / b

# Caught: the program goes on
try:
    divide(1, 0)
except ZeroDivisionError:
    print("Caught: cannot divide by zero")
print("The program goes on")

# Uncaught: the cell stops here and shows a traceback
divide(1, 0)
print("Never printed")

##### **Tracebacks**

A **traceback** is a report containing the function calls made in your code at a specific point. When your program results in an **exception,** Python will print the current traceback to help you know what went wrong.

It is a snapshot of the **call stack** (the chain of function calls that were active) at the moment of the error. For example:


In [ ]:
def sum_one(a):
    return b + 1

sum_one(1)

This traceback output has all of the information you will need to diagnose the issue:

1. **Last line**: exception **type** raised and some **info**. *What* happened.
2. **Lines above**: point out to the code that led to the exception being raised. Contains all the function calls moving from bottom to top, most **recent** to **oldest.** *Where* it happened and *how* the program got there.

Jupyter shows the same information with a different format (`Cell In[x], line y`, colors, arrows `---->`), but it is read the same way. Another example:

A Python traceback is best read from the **bottom-up.** In this example, the most recent call is the last, and the oldest the top. The last line is the error message line:


In [ ]:
def total(items):
    result = 0
    for item in items:
        result += item["price"]  # Exception raised here
    return result

def average_price(items):  # Second call: average_price()
    return total(items) / len(items)

cart = [{"price": 10}, {"price": 5}, {"cost": 3}]
print(average_price(cart))  # Oldest call: print()

#### **Handling Exceptions: The `try` Statement**

In Python, you use the `try` and `except` block to catch and handle exceptions.

##### **`try` and `except`**

The basic structure of this block is:

```python
try:
    <code to run>
except SomeError:
    <code to run when exception>
```
Python executes code following the `try` statement as a normal part of the program. The code that follows the `except` statement is the program’s response to any exceptions in the preceding try clause:

In [ ]:
for text in ["42", "forty-two", "7"]:
    try:
        number = int(text)
        print(f"{text!r} -> {number}")
    except ValueError:   # This type of exception will be seen later.
        print(f"{text!r} is not a valid integer")

Python runs the `try` block. If an exception is raised there, and its type **matches** the `except`, the rest of the `try` block is skipped and the `except` block runs. If no exception is raised, the `except` block is skipped.

If the exception does not match, the `except` does not catch it, and it keeps propagating as if there was no `try`:

In [ ]:
try:
    int(None)            # TypeError, not ValueError
except ValueError:
    print("Never runs: the except does not match")

However, you can catch **any** exception with a bare `except` statement without indicating the type of exception to be matched:

In [ ]:
try:
    1 / 0
except:
    pass  # Change into some informative message. Bad practice letting an exception pass

The exception was handled and the program did not crash. But letting an exception that occurred pass silently is bad practice.

##### **Catching a Specific Exception Type**

The except block from before will catch any exception, whether it is related or not to the operations in the `try` block. It is always best to be **specific** when you are handling an exception.

Python provides some **built-in** exceptions that you could raise in certain situations. For example:

In [ ]:
try:
    with open("file.txt") as file:
        read_data = file.read()
except FileNotFoundError:  # Handle the situation where Python does not find the requested file.
    print("Error: file not found.")

In this example, you are *only* handling the **specific scenario** where a file may not be found for reading.

##### **Accessing the Exception Object with `as`**

To access an exception **object,** you can assign it to a variable for inspection and use, typically `except Error as e`:

* `str(e)`: the error message
* `repr(e)`: the error type and arguments, useful for logs.
* `type(e).__name__`: the name of the exception class.
* `e.args`: the arguments the exception was created with.

The name `e` is **deleted** when the `except` block ends (avoids keeping the tracebacks in memory).

In [ ]:
try:
    {"a": 1}["b"]
except KeyError as e:
    print("str(e): ", str(e))
    print("repr(e):", repr(e))
    print("type:   ", type(e).__name__)
    print("args:   ", e.args)

try:
    int("abc")
except ValueError as e:            # "e" deleted after except block
    print("\nstr(e): ", str(e))
    print("args:   ", e.args)
    saved = e                      # Keep a reference for later

print("\nDoes 'e' still exist?", "e" in globals())
print("But 'saved' does:", repr(saved))

##### **Catching Multiple Exceptions**

You can have more than one function call in your `try` clause and anticipate catching various exceptions. Two ways, depending on whether each type needs a **different** or the **same** treatment:

1. **Several `except` clauses**: each one handles a type differently. Python checks them top to bottom and runs **first** that matches.
    ```python
    except <error1> as e:
     ...
    except <error2> as e:
        ...
    ...
    ```
2. **A tuple of types** in one `except`: same handling for all of them.

    ```python
    except (ValueError, TypeError) as e:     # Parentheses required when using "as"
    except ValueError, TypeError:            # Python 3.14+: allowed without "as"
    ```

In [ ]:
def to_float(value):
    try:
        return float(value)
    except ValueError:
        print(f"  {value!r}: text that is not a number")
    except TypeError:
        print(f"  {value!r}: a type that cannot be converted")
    return None

for v in ["3.14", "pi", None, [1, 2]]:
    result = to_float(v)
    print(f"{v!r} -> {result}")

In [ ]:
def safe_ratio(a, b):
    try:
        return a / b
    except (ZeroDivisionError, TypeError) as e:
        print(f"  Cannot compute {a!r} / {b!r}: {type(e).__name__}")
        return None

print(safe_ratio(10, 4))
print(safe_ratio(10, 0))
print(safe_ratio(10, "2"))

##### **`else`**

The `else` block runs only if the `try` block finished **without raising** any exception.

The `try` block should contain **only** the line(s) that **may raise** the exception you expect. Code in `else` is *not* protected by the `except` clauses, so an unexpected error there is not mistakenly caught and disguised as the error you were expecting:

In [ ]:
data = {"name": "Eduardo", "age": "21"}

# Without else: the try protects too much code
try:
    age = int(data["age"])
    print(f"Next year {data['nmae']} will be {age + 1}")     # Typo: 'nmae'
except KeyError:
    print("Missing field 'age'")     # The typo is reported as a missing 'age': misleading!

In [ ]:
# With else: the try only protects what may fail in an expected way
try:
    age = int(data["age"])
except (KeyError, ValueError):
    print("Missing or invalid 'age'")
else:
    print(f"Next year {data['nmae']} will be {age + 1}")     # The typo now raises a visible KeyError: the bug is not hidden

In short: `else` is executed **only** in the **abscence** of exceptions.

##### **`finally`**

The `finally` block runs <span style="color:rgb(17, 198, 17);">**always,**</span> whatever happens in the `try` block:

* No exception.
* An exception that is caught by an `except`.
* An exception that is **not** caught: `finally` runs first, and then the exception continues propagating.
* Even when the `try` or `except` block leaves with `return`, `break` or `continue`.

The `finally` clause is the place for [**cleanup**](#54-context-managers) after code execution: closing files, releasing connections...

In [ ]:
def process(value):
    try:
        print(f"try: processing {value!r}")
        return 100 / value
    except ZeroDivisionError:
        print("except: division by zero")
        return None
    finally:
        print("finally: always runs, even after 'return'")

print("Result:", process(4), "\n")
print("Result:", process(0), "\n")

try:
    process("x")                     # TypeError: not caught inside process()
except TypeError:
    print("TypeError caught outside, AFTER finally ran")

**Never** use `return` (or `break`/`continue`) inside `finally`: it overrides the `return` of the `try` and, worse, it silently **discards** any exception that was propagating. Since Python 3.14, the compiler emits a `SyntaxWarning`:

In [ ]:
def bad():
    try:
        raise ValueError("Important error")
    finally:
        return "finally wins"        # Swallows the exception

print(bad())                         # No error at all: the ValueError was lost

##### **Order of Execution**

The complete statement and the order in which its parts run:

```python
try:
    ...     # 1. Always, start
except A:
    ...     # 2a. Only if an exception matching A was raised in try
except B:
    ...     # 2a. Only if it matched B (and not A)
except ...:
    ...     # ...
else:
    ...     # 2b. Only if "try" finished without exceptions
finally:
    ...     # 3. Always, last
```

Rules:

* `except` clauses go first, then `else`, then `finally`. `else` requires at least one `except`.
* A `try` needs at least one `except` **OR** a `finally`.
* An exception raised **inside** an `except` or `else` block is **not** caught by the clauses of that same `try` (it propagates, after `finally` runs).

In [ ]:
def run(action):
    steps = []
    try:
        steps.append("try")
        action()
    except ValueError:
        steps.append("except")
    else:
        steps.append("else")
    finally:
        steps.append("finally")
        print(f"{action.__name__}: " + " -> ".join(steps))

def works():
    pass

def value_error():
    raise ValueError

def key_error():
    raise KeyError

run(works)          # try -> else -> finally
run(value_error)    # try -> except -> finally
try:
    run(key_error)  # try -> finally, then the KeyError propagates
except KeyError:
    print("          (the KeyError propagated after finally)")

##### **Nested `try` Blocks**

A `try` can contain another `try`, directly or through function calls. When an exception is raised, the **innermost** `try` gets the first chance to handle it. If none of its `except` clauses match, its `finally` runs and the exception moves on to the **outer** `try`.

For example:

In [ ]:
def nested(value):
    try:
        print("Outer try")
        try:
            print("  Inner try")
            number = int(value)                       # May raise ValueError
            print(f"  Inner: 1 / {number} = {1 / number}")   # May raise ZeroDivisionError
        except ValueError:
            print("  Inner except: not a number")
        finally:
            print("  Inner finally")
        print("Outer: after the inner try")
    except ZeroDivisionError:
        print("Outer except: division by zero")
    finally:
        print("Outer finally\n")

nested("4")        # No exceptions
nested("abc")      # Handled by the inner try
nested("0")        # The inner try does not handle it: it goes to the outer one

#### **Exceptions and [Context Managers](#54-context-managers)**

##### **`finally` vs `with`**

It was shown before that `with` replaces the `try` / `finally` pattern for cleanup/teardown. From the exceptions point of view:

| | `try` / `finally` | `with` |
|---|---|---|
| Who writes the cleanup | You, every time you use the resource | The context manager, once (`__exit__`) |
| Can it be forgotten? | Yes | No: it is tied to the object |
| Works with | Anything | Objects implementing the protocol |

Both **guarantee** the release of the resource: `finally` because it always runs, `with` because it always calls `__exit__` (which internally is a `try`/`finally`).

##### **Exceptions inside a `with` Block**

When an exception is raised inside a `with` block:

1. The rest of the block is skipped.
2. `__exit__` is called with the exception information, so the **resource is released**.
3. If `__exit__` returns `False`, the exception **keeps propagating** and can be caught by a `try` around the `with`. If it returns `True`, the exception is suppressed.

In other words: **`with` does not handle errors, but teardown**. The usual combination is `with` for cleanup and `try` / `except` for handling:

In [ ]:
from pathlib import Path

path = Path("numbers.txt")
path.write_text("10\n20\nthirty\n40\n", encoding="utf-8")

try:
    with open(path, encoding="utf-8") as f:
        total = 0
        for line in f:
            total += int(line)            # Fails on 'thirty'
except ValueError as e:
    print("Error while reading:", e)
    print("Is the file closed?", f.closed)   # True: 'with' closed it before reaching the except

path.unlink()                             # Delete the example file

### 6.2 The Exception Hierarchy

#### **Exception Classes**

##### **`BaseException` and `Exception`**

Exceptions are **classes,** and they are organized in an **[inheritance](#inheritance) hierarchy**. An abbreviated version of the built-in one is:

```c++
BaseException
 ├── BaseExceptionGroup
 ├── GeneratorExit
 ├── KeyboardInterrupt
 ├── SystemExit
 └── Exception
      ├── ArithmeticError
      │    ├── OverflowError
      │    └── ZeroDivisionError
      ├── AssertionError
      ├── AttributeError
      ├── ImportError
      │    └── ModuleNotFoundError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── NameError
      ├── OSError
      │    ├── FileExistsError
      │    ├── FileNotFoundError
      │    ├── PermissionError
      │    └── TimeoutError
      ├── RuntimeError
      │    ├── NotImplementedError
      │    └── RecursionError
      ├── StopIteration
      ├── TypeError
      ├── ValueError
      │    └── UnicodeError
      └── Warning
```
In Python, all exceptions must be instances of a class that derives from `BaseException`. It is the base class of all **built-in** exceptions. Its direct subclasses other than `Exception` are not really errors, but **signals to stop**:
* `SystemExit`: raised by `sys.exit()` (terminates program).
* `KeyboardInterrupt`: the user pressed `Ctrl+C` (or the stop button in Jupyter).
* `GeneratorExit`: raised inside a generator when it is closed (`.close()`).

**`Exception`** is the the base class of all "normal" errors. It is also the class **your own exceptions** should inherit from.

You can explore the hierarchy with OOP tools:

In [ ]:
print(KeyError.__mro__)
print(FileNotFoundError.__mro__)
print()
print(issubclass(ZeroDivisionError, ArithmeticError))   # True
print(issubclass(KeyboardInterrupt, Exception))         # False
print(issubclass(KeyboardInterrupt, BaseException))     # True

In [ ]:
def print_tree(cls, max_depth, depth=0):
    print("    " * depth + cls.__name__)
    if depth < max_depth:
        for sub in cls.__subclasses__():
            print_tree(sub, max_depth, depth + 1)

print_tree(BaseException, max_depth=1)
print()
print_tree(ArithmeticError, max_depth=1)
print()
print_tree(OSError, max_depth=1)       # Libraries can add their own subclasses, so you may see more

##### **General vs Specific Exceptions**

`except E:` catches `E` **and all its subclasses**: the same rule as `isinstance(exc, C)` from OOP. The higher the class in the tree, the more exceptions it catches:

* **Specific** (`KeyError`, `FileNotFoundError`): precise handling of a well-known situation.
* **General** (`ArithmeticError`, `OSError`): one handler for a whole family of related errors.

In [ ]:
def get_item(container, key):
    try:
        return container[key]
    except LookupError as e:          # Catches KeyError AND IndexError
        print(f"{type(e).__name__} caught as LookupError")
        return None

get_item({"a": 1}, "b")      # KeyError
get_item([1, 2, 3], 10)      # IndexError

So, in a `try` statement with an `except` clause that mentions a particular exception class, that clause also handles any exception classes **derived** from that class (but not exception classes from which it is derived). 

##### **Catching `Exception` vs `BaseException`**

`KeyboardInterrupt` and `SystemExit` are not errors in your code: they are **requests to stop the program**. That is precisely why they do *not* inherit from `Exception`: so that `except Exception` catches every *error*, but lets these signals pass through.

If you catch `BaseException` (or use a bare `except:`, which is the same), `Ctrl+C` stops working inside that block and `sys.exit()` gets ignored.

In [ ]:
import sys

def main():
    try:
        sys.exit("Program finished by the user")      # Raises SystemExit
    except Exception:
        print("Never runs: SystemExit is not an Exception")

try:
    main()
except BaseException as e:            # Only for this demo: to see what got through
    print(f"{type(e).__name__} went through 'except Exception': {e}")

In [ ]:
def stubborn():
    try:
        sys.exit("Please, stop")
    except BaseException:              # Too broad: it swallows the exit request
        print("Exit request ignored, the program keeps going...")

stubborn()

#### **Common Built-in Exceptions**

| Exception | When it is raised | Example |
|---|---|---|
| `ValueError` | Right type, inappropriate **value** | `int("abc")`, `math.sqrt(-1)` |
| `TypeError` | Operation applied to the wrong **type** | `"a" + 1`, `len(5)` |
| `KeyError` | Missing dictionary key | `{}["x"]` |
| `IndexError` | Sequence index out of range | `[1, 2][5]` |
| `AttributeError` | The object has no such attribute/method | `None.upper()` |
| `NameError` | Name not defined (often a typo) | `pritn("hi")` |
| `ZeroDivisionError` | Division or modulo by zero | `1 / 0` |
| `FileNotFoundError` | (`OSError`) The file does not exist | `open("missing.txt")` |
| `PermissionError` | (`OSError`) Not allowed to access it | Writing to a protected folder |
| `ModuleNotFoundError` | (`ImportError`) The module cannot be found | `import nonexistent` |
| `StopIteration` | The iterator has no more elements (5.1) | `next(iter([]))` |
| `RecursionError` | Maximum recursion depth exceeded | A recursive function without base case |
| `NotImplementedError` | A method that subclasses must implement (4.5) | `raise NotImplementedError` |
| `AssertionError` | An `assert` condition is false | `assert 1 > 2` |
| `RuntimeError` | Generic error that fits no other category | |

In [ ]:
import math

examples = {
    'int("abc")':          lambda: int("abc"),
    'math.sqrt(-1)':       lambda: math.sqrt(-1),
    '"a" + 1':             lambda: "a" + 1,
    'len(5)':              lambda: len(5),
    '{}["x"]':             lambda: {}["x"],
    '[1, 2][5]':           lambda: [1, 2][5],
    'None.upper()':        lambda: None.upper(),
    'pritn("hi")':         lambda: pritn("hi"),
    '1 / 0':               lambda: 1 / 0,
    'open("missing.txt")': lambda: open("missing.txt"),
    'import nonexistent':  lambda: __import__("nonexistent"),
    'next(iter([]))':      lambda: next(iter([])),
}

for expression, func in examples.items():
    try:
        func()
    except Exception as e:          # Broad on purpose: we want to see every type
        print(f"{expression:<20} -> {type(e).__name__}: {e}")

#### **Order of `except` Clauses**

Python checks the `except` clauses **from top to bottom** and runs only the **first** one that matches. Since a general class matches all its subclasses, the **specific exceptions must go first**; otherwise, the specific clauses are unreachable:

In [ ]:
def read_config(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except OSError:                         # General first...
        print("  OSError: some I/O problem")
    except FileNotFoundError:               # ...so this one is unreachable
        print("  FileNotFoundError: never runs")

read_config("does_not_exist.cfg")

In [ ]:
def read_config(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:               # Specific first
        print(f"  {path} does not exist, using the default configuration")
    except PermissionError:
        print(f"  No permission to read {path}")
    except OSError as e:                    # General last: any other I/O error
        print(f"  Unexpected I/O error: {e}")

read_config("does_not_exist.cfg")

#### **Bad Practices**

**1. Bare `except:`.** It is equivalent to `except BaseException:`: it catches **everything**, including `KeyboardInterrupt` and `SystemExit` (the user cannot stop the program) and every bug.

In [ ]:
for attempt in range(3):
    try:
        raise KeyboardInterrupt      # Simulates the user pressing Ctrl+C
    except:                          # Bare except: swallows it
        print(f"Attempt {attempt}: Ctrl+C ignored")

**2. Silencing errors: `except Exception: pass`**. The program goes on with a wrong state and nobody will ever know: the error appears later, far from its cause, or never (wrong results). If an error can be ignored, catch the **specific** type and at least **record** it:

In [ ]:
rows = ["10", "20", "3O", "40"]      # "3O" has a letter O, not a zero

total = 0
for row in rows:
    try:
        total += int(row)
    except Exception:
        pass                         # Silenced: nobody will know a row was lost
print("Total:", total)               # 70. Looks right... but it is wrong

total, invalid = 0, []
for row in rows:
    try:
        total += int(row)
    except ValueError:
        invalid.append(row)          # Handled: the problem is recorded
print("Total:", total, "| Invalid rows:", invalid)

**3. Giant `try` blocks**. If the `try` wraps many lines, you may not know which one raised, and you catch errors from lines you never intended to protect (as in the `else` example). Keep `try` as small as possible, and use `else` for the code that depends on it:

```python
# Bad
try:
    config = load_config(path)
    user = find_user(config, name)
    send_email(user.email, message)
except Exception:
    print("Something failed")          # What? Where?

# Better
try:
    config = load_config(path)
except FileNotFoundError:
    config = DEFAULT_CONFIG            # The only situation we know how to handle
user = find_user(config, name)         # Unexpected errors here will surface with their traceback
send_email(user.email, message)
```

### 6.3 Raising and Propagating Exceptions

There are scenarios where you might want to stop your program by <span style="color: #e90707;">**raising**</span> an exception if a condition occurs.

#### **Raising Exceptions**

##### **The `raise` Statement**

The `raise` statement allows you to force a specified exception to occur. It has a very simple strcture:

```python
raise Error                    # Raise an exception class: no arguments
raise Error("Error message")   # Raise an instance 
```
The message should say **what** is wrong and include the **offending value,** so whoever reads the traceback does not need to guess:

In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError(f"Invalid age: {age}. It cannot be negative")
    return age

try:
    set_age(-5)
except ValueError as e:
    print("Error:", e)

##### **Validating Arguments: `ValueError` vs `TypeError`**


* **`TypeError`**: raised when an operation or function is applied to an object of inappropriate type.
* **`ValueError`**: raised when an operation or function receives an argument that has the right type but an inappropriate value.

Validate at the **beginning** of the function (*guard clauses*): the error is raised close to its cause, with a clear message, instead of failing later inside some unrelated line with a confusing one.

In [ ]:
def create_account(owner, initial_balance=0):
    if not isinstance(owner, str):
        raise TypeError(f"Owner must be a str, not {type(owner).__name__}")
    if not owner.strip():
        raise ValueError("Owner cannot be empty")
    if isinstance(initial_balance, bool) or not isinstance(initial_balance, (int, float)):
        raise TypeError(f"initial_balance must be a number, not {type(initial_balance).__name__}")
    if initial_balance < 0:
        raise ValueError(f"initial_balance cannot be negative: {initial_balance}")
    return {"Owner": owner, "Balance": initial_balance}

tests = [("Eduardo", 100), (42, 100), ("   ", 100), ("Marcos", "100"), ("X", -50)]
for args in tests:
    try:
        print(create_account(*args))
    except (TypeError, ValueError) as e:
        print(f"{type(e).__name__}: {e}")

##### **Returning a Special Value vs Raising**

The alternative to raising is returning a special value (`None`, `-1`, `False`) to mean "it failed". It has problems though:

* The caller can **forget to check** it, and the program continues with a wrong value: the error appears later, far from its cause.
* The special value may be a **valid result** (`0`, `None`, `-1`...).
* It does not say **why** it failed.

An exception cannot be ignored by accident: either someone handles it, or the program stops and tells you where. For example:

In [ ]:
text = "Hello World"

pos = text.find("z")            # -1 means "not found"...
print(pos, repr(text[pos]))     # ...but -1 is a valid index: silently gives 'd'

try:
    pos = text.index("z")
except ValueError as e:
    print("index():", e)        # The problem cannot go unnoticed

A simple guideline for a good practice would be:

* **Return** a **special** value when "not found / no result" is a **normal, expected** outcome and the value is unambiguous: `dict.get(key)` → `None`; `str.find()` → `-1`.
* **Raise** when a function **cannot** do what it promises/it is supposed to do.

#### **Propagation**

##### **The Call Stack**

Each function call creates a **frame** (its local variables and the point where it is executing), which is pushed onto the **call stack.** When the function returns, its frame is removed. When an exception is raised and the current function does not catch it:

1. The function is **abandoned** at that line (but its `finally` blocks run).
2. The exception is passed to the **caller**, at the line where the call was made.
3. The process repeats up the stack until some `try` catches it, or it reaches the top level (uncaught → traceback).

The **traceback** is exactly the record of that path.

In [ ]:
def level_3():
    print("    level_3: start")
    raise ValueError("Problem in level_3")
    print("    level_3: end")            # Never runs

def level_2():
    print("  level_2: start")
    try:
        level_3()
    finally:
        print("  level_2: finally (cleanup on the way up)")
    print("  level_2: end")              # Never runs

def level_1():
    print("level_1: start")
    try:
        level_2()
    except ValueError as e:
        print(f"level_1: caught '{e}'")
    print("level_1: end")                # Runs: the exception was handled here

level_1()

The `traceback` module lets you print (or get as text) the traceback of the exception being handled, **without** stopping the program. It is useful to log (registrar) errors:

In [ ]:
import traceback

try:
    level_2()
except ValueError:
    print("\n")
    traceback.print_exc()        # Prints the full traceback (to stderr)

print("The program is still running")

##### **Re-raising with a Bare `raise`**

Inside an `except` block, `raise` **without arguments** re-raises the exception being handled, unchanged and with its original traceback. Use it when you want to **do something** with the error (record it, add information) but you are not the right place to **resolve** it, so you let it keep going up.

In [ ]:
def load_user(users, user_id):
    try:
        return users[user_id]
    except KeyError:
        print(f"[log] User {user_id!r} not found")     
        raise                                          # ...and let it go on

users = {1: "A", 2: "B"}
try:
    load_user(users, 3)
except KeyError as e:
    print("The caller handles it:", repr(e))

##### **Where to Catch an Exception**

Catch an exception **only where you can do something useful with it.** If you cannot do anything useful, **let it propagate**. 

Low-level functions usually *raise*; the high-level code, which knows the context, *decides*. For example:

In [ ]:
import random

random.seed(7)

def fetch_temperature():                                    # Raises
    """Simulates an unreliable sensor (or network call)."""
    if random.random() < 0.6:
        raise ConnectionError("Sensor not responding")
    return round(random.uniform(15, 25), 1)

def read_temperature(retries=3, default=None):              # Handles/decides
    for attempt in range(1, retries + 1):
        try:
            return fetch_temperature()
        except ConnectionError as e:          # Useful here: we can retry
            print(f"  Attempt {attempt} failed: {e}")
    return default                            # And then fall back to a default

for _ in range(3):
    print("Temperature:", read_temperature())

#### **Exception Chaining**

##### **Implicit Chaining: `__context__`**

If an exception is raised **while another one is being handled** (inside an `except` or `finally` block), Python does not lose the first one: it stores it in the `__context__` attribute of the new one, and the traceback shows both, separated by:

```python
"During handling of the above exception, another exception occurred:"
```

It usually means that there is a **bug** in the handler (`except`):

In [ ]:
config = {"timeout": "abc"}

try:
    timeout = int(config["timeout"])
except ValueError:
    timeout = int(config["default_timeout"])     # Bug in the handler: KeyError

In [ ]:
try:
    try:
        int("abc")
    except ValueError:
        {}["missing"]
except KeyError as e:
    print("Exception:  ", repr(e))
    print("__context__:", repr(e.__context__))
    print("__cause__:  ", repr(e.__cause__))

##### **Explicit Chaining: `raise ... from ...`**

Sometimes you **deliberately** replace a low-level exception with a more meaningful one. Using `raise NewError(...) from e` stores the original in the `__cause__` attribute of the new one, and the traceback shows both, now separated by:

```python
"The above exception was the direct cause of the following exception:"
```

which says "this was intentional", not a bug in the handler. You get a clear high-level error without losing the low-level detail for debugging.

In [ ]:
def load_port(config):
    try:
        return int(config["port"])
    except (KeyError, ValueError) as e:
        raise RuntimeError("Invalid configuration: 'port' must be an integer") from e

load_port({"port": "eighty"})

In [ ]:
try:
    load_port({})
except RuntimeError as e:
    print("Exception:  ", repr(e))
    print("__cause__:  ", repr(e.__cause__))
    print("__context__:", repr(e.__context__))      # Also set

##### **Hiding the Cause: `from None`**

`raise NewError(...) from None` hides the previous exception from the traceback (it sets the attribute `__suppress_context__ = True`; the original is still stored in `__context__`). Use it when the original exception is an **implementation detail** that only adds noise. For example:

In [ ]:
class Settings:
    def __init__(self, **values):
        self._values = values

    def __getattr__(self, name):
        try:
            return self._values[name]
        except KeyError:
            raise AttributeError(f"Setting {name!r} does not exist") from None

settings = Settings(theme="dark")
print(settings.theme)
settings.language  # Only the AttributeError appears: the internal KeyError (there's no "language" key) is hidden

| How | `__context__` | `__cause__` | Traceback |
|---|---|---|---|
| Error inside an `except` (implicit) | original | `None` | *"During handling of the above exception, another exception occurred"* |
| `raise New(...) from e` | original | `e` | *"The above exception was the direct cause of the following exception"* |
| `raise New(...) from None` | original (hidden) | `None` | Only the new exception |

##### **Adding Information with `add_note()`**

For Python 3.11+, the method `e.add_note("...")` attaches extra info. to an exception, shown in the traceback below the message, **without** changing its **type.** Combined with a bare `raise`, it is a clean way to add context on the way up:

In [ ]:
def parse_rows(rows):
    numbers = []
    for number, row in enumerate(rows, start=1):
        try:
            numbers.append(int(row))
        except ValueError as e:
            e.add_note(f"While parsing row {number}: {row!r}")
            raise
    return numbers

parse_rows(["1", "2", "x", "4"])

### 6.4 User-Defined (Custom) Exceptions

Built-in exceptions describe *technical* problems (wrong value, missing key...). Programs may also have errors of their own **type:** invalid file format, a user without permissions... Custom exceptions let you represent them, and since they are classes, everything from OOP applies.

#### **Creating Exceptions**

##### **Inheriting from `Exception`**

Programs may name their own exceptions by creating a new exception class. Exceptions should typically be derived from the **`Exception`** class, either directly or indirectly (from another built-in exception). Everything is inherited: message, `args`, traceback, chaining, `add_note()`...

Conventions:

* Inherit from **`Exception`**, never from `BaseException`: the statement `except Exception` must catch it.
* Most exceptions are defined with names that end in “Error”, similar to the naming of the standard exceptions.
  
A simple custom exception would be:

In [ ]:
class InsufficientFundsError(Exception):
    """Raised when an account does not have enough money for an operation."""

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(f"Cannot withdraw {amount}: the balance is {balance}")
    return balance - amount

try:
    withdraw(50, 80)
except InsufficientFundsError as e:
    print(f"{type(e).__name__}: {e}")
    print("Is it an Exception?", isinstance(e, Exception))

While you can customize your exception object, you do not need to do that. It is often enough to give your custom Python exceptions a descriptive name and a **docstring.**

##### **Custom Messages and Attributes**

An exception can carry **structured data** as attributes so the handler can **act** on it instead of parsing the message text. It is done by defining `__init__`, storing the attributes, and calling `super()` so that `str(e)` and `e.args` still work:

In [ ]:
class InsufficientFundsError(Exception):
    """Raised when an account does not have enough money for an operation."""

    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        self.missing = amount - balance
        super().__init__(f"Cannot withdraw {amount}: the balance is {balance} (missing {self.missing})")


class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError(f"The amount must be positive: {amount}")   # A built-in fits perfectly here
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount


account = Account("Ana", 50)
try:
    account.withdraw(80)
except InsufficientFundsError as e:
    print("Message:", e)
    print(f"Would you like a loan of {e.missing}?")    # The handler uses the data, not the text

#### **Custom Hierarchies**

##### **A Base Exception for Your Project**

The usual design of an exception hierarchy, used by most libraries, is:

1. **One base exception** for the whole project or module.
2. **Specific subclasses** for each kind of problem.

A subclass can **also** inherit from a built-in, so that code that only knows the built-ins can still catch it.

For example, a module that loads data from rows of text:

In [ ]:
class DataError(Exception):
    """Base class for every error of the data loading module."""


class InvalidFormatError(DataError):
    """A row does not have the expected structure."""

    def __init__(self, row_number, row):
        self.row_number = row_number
        self.row = row
        super().__init__(f"Row {row_number}: invalid format {row!r}")


class OutOfRangeError(DataError, ValueError):
    """A value is outside its allowed range. It is also a ValueError."""

    def __init__(self, row_number, field, value, low, high):
        self.row_number = row_number
        self.field = field
        self.value = value
        super().__init__(f"Row {row_number}: {field}={value} is outside [{low}, {high}]")


print([cls.__name__ for cls in OutOfRangeError.__mro__], "\n")  # Exception Hierarchy

def parse_row(row_number, row):
    """Parse a 'letter, number' row into a dictionary."""
    parts = row.split(",")
    if len(parts) != 2:
        raise InvalidFormatError(row_number, row)

    letter, number = (part.strip() for part in parts)
    try:
        number = int(number)
    except ValueError as e:
        raise InvalidFormatError(row_number, row) from e    # Translate into a domain error, keeping the cause

    if not 0 <= number <= 120:
        raise OutOfRangeError(row_number, "number", number, 0, 120)
    return {"letter": letter, "number": number}

print(parse_row(1, "A, 3"))

##### **Catching Individually or Through the Base Class**

With this hierarchy, each caller chooses the level of detail it needs:

In [ ]:
rows = ["A, 31", "B;40", "C, 150", "D, abc", "E, 28"]

# 1. Catch the specific cases to handle each one differently
series = []
for n, row in enumerate(rows, start=1):
    try:
        series.append(parse_row(n, row))
    except OutOfRangeError as e:
        print(f"  [Range]  {e} -> field {e.field!r}, value {e.value}")
    except InvalidFormatError as e:
        print(f"  [Format] {e}")
print("Valid:", series, "\n")

# 2. Catch the whole group through the base class
errors = []
for n, row in enumerate(rows, start=1):
    try:
        parse_row(n, row)
    except DataError as e:
        errors.append(e)
print(f"{len(errors)} rows with problems:", [type(e).__name__ for e in errors], "\n")

# 3. Code that only knows the built-ins still works: OutOfRangeError is a ValueError
try:
    parse_row(1, "A, -3")
except ValueError as e:
    print("Caught as a ValueError:", e)

### 6.5 Style: EAFP vs LBYL

#### **Two Approaches**

There are two ways of dealing with an operation that may fail:

* **LBYL,** *"Look Before You Leap"*: check the preconditions first, and only then do the operation.
  ```python
  if key in d:
      value = d[key]
  ```
* **EAFP,** *"Easier to Ask for Forgiveness than Permission"*: do the operation directly, and handle the exception if it fails.
  ```python
  try:
      value = d[key]
  except KeyError:
      ...
  ```

This common Python coding style assumes the existence of valid **keys** or **attributes** and catches exceptions if the assumption proves false, as the [glossary](https://docs.python.org/3/glossary.html#term-EAFP) says. Both are correct; the point is to know when each one is clearer.

In [ ]:
stock = {"apples": 10, "pears": 0}

def get_stock_lbyl(product):
    if product in stock:              # Look...
        return stock[product]         # ...then leap
    return 0

def get_stock_eafp(product):
    try:
        return stock[product]         # Just do it...
    except KeyError:                  # ...and "apologize" if it fails
        return 0

print(get_stock_lbyl("apples"), get_stock_lbyl("kiwis"))
print(get_stock_eafp("apples"), get_stock_eafp("kiwis"))

# For this simple case, the best option is neither: dict.get() does exactly this
print(stock.get("apples", 0), stock.get("kiwis", 0))

#### **Why Python Favours EAFP**

**1. The operation is its own best check.** Writing the check means re-implementing the rules of the operation, and it is easy to get them wrong. For example:

In [ ]:
# Example Operation: convert string number to int

def parse_lbyl(text):
    if text.isdigit():                # Seems reasonable, but innecessary check. 
        return int(text)
    return None

def parse_eafp(text):
    try:
        return int(text)              # The operation is its own check.
    except ValueError:                # Does not matter if it fails. We are able to catch it!
        return None

for t in ["42", "-7", " 8 ", "1_000", "abc"]:
    print(f"{t!r:>8}: LBYL={parse_lbyl(t)!s:<5} EAFP={parse_eafp(t)}")

**2. Race conditions.** Between the check and the action, the world can change: another program can delete the file, another thread can remove the key. The check was true, but it no longer is (a *time-of-check to time-of-use* bug). EAFP does not have that gap:

In [ ]:
import os

# LBYL: the file may be deleted right between both lines
if os.path.exists(path):
    with open(path) as f:
        data = f.read()

# EAFP: a single operation, no gap
try:
    with open(path) as f:
        data = f.read()
except FileNotFoundError:
    data = ""

**3. Duck typing.** Instead of checking `isinstance(obj, Class)`, just use the object, and handle `AttributeError`/`TypeError` if it does not support the operation.

**4. Performance.** Entering a `try` costs practically nothing when no exception is raised; *raising and catching* one is much more expensive than an `if`. So EAFP is efficient when failure is **rare**, and LBYL is faster when failure is **frequent**:

In [ ]:
from timeit import timeit

d = {"a": 1}

def eafp(key):
    try:
        return d[key]
    except KeyError:
        return None

def lbyl(key):
    if key in d:
        return d[key]
    return None

n = 500_000
print(f"Key present -> EAFP: {timeit(lambda: eafp('a')):.3f} s   LBYL: {timeit(lambda: lbyl('a')):.3f} s")
print(f"Key missing -> EAFP: {timeit(lambda: eafp('z')):.3f} s   LBYL: {timeit(lambda: lbyl('z')):.3f} s")

In the end, **LBYL** is clearer when:

* The **"failure"** is a **normal** case, not an exceptional one: if most keys will be missing, an `if` expresses it better (and is faster).
* The check is **cheap,** obvious: `if not x:` (empty), `if y == 0:`...
* A **`try`** statement would have to be **big** and the exception could come from somewhere else: a precise check is safer than a broad `except`.

Rule of thumb: **EAFP** when failure is exceptional and the operation itself is the best check; **LBYL** when failure is the normal path or a partial action would be harmful.

For more information: [Errors and Exceptions (Python tutorial)](https://docs.python.org/3/tutorial/errors.html), [Built-in Exceptions and their hierarchy](https://docs.python.org/3/library/exceptions.html#exception-hierarchy), [The `try` statement](https://docs.python.org/3/reference/compound_stmts.html#the-try-statement), [PEP 3134 – Exception Chaining](https://peps.python.org/pep-3134/), [PEP 678 – Exception Notes](https://peps.python.org/pep-0678/), [PEP 765 – Disallow return/break/continue in finally](https://peps.python.org/pep-0765/), [EAFP (Python glossary)](https://docs.python.org/3/glossary.html#term-EAFP), [`traceback`](https://docs.python.org/3/library/traceback.html).